In [33]:
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
import os, glob, json
import numpy as np
import pandas as pd

from skimage.io import imread
from skimage.color import rgb2gray
from skimage.measure import label, regionprops
from skimage.morphology import binary_opening, binary_closing, remove_small_objects, disk

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score


In [35]:
ROOT = "/content/drive/MyDrive/ML_MED/Practical_2/"
TRAIN_IMG_DIR = os.path.join(ROOT, "training_set")
TEST_IMG_DIR  = os.path.join(ROOT, "test_set")
TRAIN_CSV = os.path.join(ROOT, "training_set_pixel_size_and_HC.csv")
TEST_CSV  = os.path.join(ROOT, "test_set_pixel_size.csv")

OUT_DIR = os.path.join(ROOT, "outputs_rf")
os.makedirs(OUT_DIR, exist_ok=True)

df_train_meta = pd.read_csv(TRAIN_CSV)
df_test_meta  = pd.read_csv(TEST_CSV)

print("TRAIN_IMG_DIR exists:", os.path.exists(TRAIN_IMG_DIR))
print("TEST_IMG_DIR exists:", os.path.exists(TEST_IMG_DIR))
print("TRAIN_CSV exists:", os.path.exists(TRAIN_CSV))
print("TEST_CSV exists:", os.path.exists(TEST_CSV))
print("Train meta shape:", df_train_meta.shape)
print("Test meta shape:", df_test_meta.shape)


TRAIN_IMG_DIR exists: True
TEST_IMG_DIR exists: True
TRAIN_CSV exists: True
TEST_CSV exists: True
Train meta shape: (999, 3)
Test meta shape: (335, 2)


In [36]:
def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

id_col_train = pick_col(df_train_meta, ["id","ID","image","image_id","Image","filename","file","img"])
id_col_test  = pick_col(df_test_meta,  ["id","ID","image","image_id","Image","filename","file","img"])
if id_col_train is None: id_col_train = df_train_meta.columns[0]
if id_col_test is None:  id_col_test  = df_test_meta.columns[0]

pix_col_train = pick_col(df_train_meta, ["pixel size(mm)","pixel size (mm)","pixel_size","pixel_size(mm)","pixel_size_mm"])
pix_col_test  = pick_col(df_test_meta,  ["pixel size(mm)","pixel size (mm)","pixel_size","pixel_size(mm)","pixel_size_mm"])
if pix_col_train is None or pix_col_test is None:
    raise ValueError("Pixel size column not found in CSV files.")

target_col = pick_col(df_train_meta, ["head circumference (mm)","HC","hc","head_circumference_mm"])
if target_col is None:
    raise ValueError("Target column (HC) not found in training CSV.")

print("id_col_train:", id_col_train)
print("id_col_test :", id_col_test)
print("pix_col_train:", pix_col_train)
print("pix_col_test :", pix_col_test)
print("target_col:", target_col)


id_col_train: filename
id_col_test : filename
pix_col_train: pixel size(mm)
pix_col_test : pixel size(mm)
target_col: head circumference (mm)


In [37]:
def to_binary(mask):
    if mask.ndim == 3:
        mask = rgb2gray(mask)
    m = mask.astype(np.float32)
    if m.max() > 1.0:
        m = m / 255.0
    m = m > 0.5
    m = binary_opening(m, disk(1))
    m = binary_closing(m, disk(2))
    m = remove_small_objects(m, 30)
    return m

def largest_props(binary_mask):
    lab = label(binary_mask)
    props = regionprops(lab)
    if len(props) == 0:
        return None
    props.sort(key=lambda p: p.area, reverse=True)
    return props[0]

def find_mask_path(img_dir, image_id):

    image_id = os.path.splitext(image_id)[0]

    cands = []

    search_patterns = [
        f"{image_id}_Annotation.png",
        f"{image_id}_mask.png",
        f"{image_id}.png",
        f"{image_id}*.png",
        f"{image_id}*.jpg",
        f"{image_id}*.bmp"
    ]

    for pattern in search_patterns:
        found = glob.glob(os.path.join(img_dir, pattern))
        cands.extend(found)


    cands = sorted(list(set(cands)))

    if len(cands) == 0:
        return None


    for p in cands:
        name = os.path.basename(p).lower()
        if "annotation" in name or "mask" in name or "seg" in name:
            return p


    return cands[0]


In [38]:
import os


train_cache = os.path.join(OUT_DIR, "train_features.csv")
test_cache  = os.path.join(OUT_DIR, "test_features.csv")

if os.path.exists(train_cache):
    os.remove(train_cache)
    print("Đã xóa cache cũ:", train_cache)

if os.path.exists(test_cache):
    os.remove(test_cache)
    print("Đã xóa cache cũ:", test_cache)

print("Bây giờ hãy chạy lại Cell build_feature_table!")

Đã xóa cache cũ: /content/drive/MyDrive/ML_MED/Practical_2/outputs_rf/train_features.csv
Đã xóa cache cũ: /content/drive/MyDrive/ML_MED/Practical_2/outputs_rf/test_features.csv
Bây giờ hãy chạy lại Cell build_feature_table!


In [39]:
def build_feature_table(df_meta, img_dir, id_col, pix_col, cache_path):
    if os.path.exists(cache_path):
        feat = pd.read_csv(cache_path)
        return feat, 0
    feats = []
    missing = 0
    for _, row in df_meta.iterrows():
        image_id = str(row[id_col])
        ps = float(row[pix_col])
        mp = find_mask_path(img_dir, image_id)
        if mp is None or (not os.path.exists(mp)):
            missing += 1
            f = extract_features(None, ps)
        else:
            f = extract_features(mp, ps)
        f["image_id"] = image_id
        f["pixel_size"] = ps
        feats.append(f)
    feat = pd.DataFrame(feats)
    feat.to_csv(cache_path, index=False)
    return feat, missing

train_cache = os.path.join(OUT_DIR, "train_features.csv")
test_cache  = os.path.join(OUT_DIR, "test_features.csv")

train_feat, miss_train = build_feature_table(df_train_meta, TRAIN_IMG_DIR, id_col_train, pix_col_train, train_cache)
test_feat,  miss_test  = build_feature_table(df_test_meta,  TEST_IMG_DIR,  id_col_test,  pix_col_test,  test_cache)

print("Missing train masks:", miss_train)
print("Missing test masks :", miss_test)
print("Train features shape:", train_feat.shape)
print("Test features shape :", test_feat.shape)


Missing train masks: 0
Missing test masks : 0
Train features shape: (999, 20)
Test features shape : (335, 20)


In [40]:
df_train_all = train_feat.merge(df_train_meta[[id_col_train, target_col]], left_on="image_id", right_on=id_col_train, how="left")
y = df_train_all[target_col].astype(float)

X = df_train_all.drop(columns=[target_col, id_col_train, "image_id"])
X_test = test_feat.drop(columns=["image_id"])

keep_cols = [c for c in X.columns if c.endswith("_mm") or c.endswith("_mm2") or c in ["eccentricity","solidity","extent","orientation"]]
X = X[keep_cols]
X_test = X_test[keep_cols]

print("X shape:", X.shape)
print("X_test shape:", X_test.shape)


X shape: (999, 11)
X_test shape: (335, 11)


In [42]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(random_state=42, n_jobs=-1)

param_dist = {
    "n_estimators": [200, 500, 800, 1200],
    "max_depth": [None, 10, 20, 30, 40],
    "max_features": ["sqrt", "log2", 0.7, 0.9],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8],
    "bootstrap": [True]
}

search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    n_iter=40,
    scoring="neg_mean_absolute_error",
    cv=5,
    random_state=42,
    n_jobs=-1,
    return_train_score=True
)

search.fit(X_train, y_train)

best_rf = search.best_estimator_
best_params = search.best_params_
best_cv_mae = -search.best_score_

val_pred = best_rf.predict(X_val)
val_mae = mean_absolute_error(y_val, val_pred)
val_r2 = r2_score(y_val, val_pred)

print("Best params:", best_params)
print("Best CV MAE:", best_cv_mae)
print("Hold-out MAE:", val_mae)
print("Hold-out R2:", val_r2)

pd.DataFrame(search.cv_results_).to_csv(os.path.join(OUT_DIR, "rf_cv_results.csv"), index=False)
with open(os.path.join(OUT_DIR, "best_params.json"), "w") as f:
    json.dump(best_params, f)

metrics_txt = "\n".join([
    f"Best CV MAE: {best_cv_mae}",
    f"Hold-out MAE: {val_mae}",
    f"Hold-out R2: {val_r2}",
    f"Best params: {best_params}"
])
with open(os.path.join(OUT_DIR, "metrics.txt"), "w") as f:
    f.write(metrics_txt)

print("Saved:", os.path.join(OUT_DIR, "rf_cv_results.csv"))
print("Saved:", os.path.join(OUT_DIR, "best_params.json"))
print("Saved:", os.path.join(OUT_DIR, "metrics.txt"))


Best params: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 0.9, 'max_depth': 10, 'bootstrap': True}
Best CV MAE: 7.385225391811292
Hold-out MAE: 7.830979404139493
Hold-out R2: 0.9700778488328335
Saved: /content/drive/MyDrive/ML_MED/Practical_2/outputs_rf/rf_cv_results.csv
Saved: /content/drive/MyDrive/ML_MED/Practical_2/outputs_rf/best_params.json
Saved: /content/drive/MyDrive/ML_MED/Practical_2/outputs_rf/metrics.txt


In [43]:
best_rf.fit(X, y)
test_pred = best_rf.predict(X_test)

submission = pd.DataFrame({"HC": test_pred})
sub_path = os.path.join(OUT_DIR, "rf_submission.csv")
submission.to_csv(sub_path, index=False)

print("Saved submission:", sub_path)


Saved submission: /content/drive/MyDrive/ML_MED/Practical_2/outputs_rf/rf_submission.csv
